# MIF / MIF-ST / CARP-640M scoring for decoding-design-bias

**This notebook supersedes cell 17 of `Calculate_All_models_likelihoods_ORIGINAL.ipynb`
for any future scoring run.** The master-notebook cell remains in place as the historical
reference (it produced the existing v4 `mif_score` / `mifst_score` / `carp_640M_score`
columns) but it does **not** apply masked-marginal masking — see
[`AUDIT_REPORT.md`](AUDIT_REPORT.md) for the CRITICAL finding.

This notebook fixes that.

## What changes vs the master-notebook cell

The master notebook does a single forward pass with the WT sequence as input, then
gathers `log P(WT_i | full WT sequence)` at every position. The model sees its own
answer, so the score is inflated relative to the masked-marginal PLL that the methods
section claims.

This notebook does **proper masked-marginal pseudo-log-likelihood**, the same way
ESM2 does it in cell 23 of the master notebook:

- For each position `i` in the WT sequence, replace token `i` with the model's mask
  token, leave every other position intact, forward pass.
- Read `log p(WT_i | masked-context)` from the output logits at position `i`.
- Average over all positions to get the per-residue mean log p.

Positions are batched per forward pass (`MASK_BATCH_SIZE`) for speed. Validation
against a single-position reference loop is included in §4.

## Scoring convention (matches the rest of the pipeline)

- `<model>_score`     : per-residue mean masked log p (higher = better, ≤ 0)
- `<model>_sum`       : sum of per-position masked log p
- `<model>_positions` : number of positions actually scored

Sign convention: log-softmax outputs, values ≤ 0.

## Sequence-cleaning policy (matches ESM2 / ProGen2 / ProtGPT2)

20 standard amino acids only; terminal `*` stripped; no replacement of X / B / Z / U / O.
Proteins containing non-standard residues are recorded with NaN scores and
`sequence_filter_status="nonstandard_amino_acid"` so the cohort actually scored is
directly comparable to ESM2's cohort.

## Models supported

- `mif`        : structure-conditioned (needs AlphaFold PDB)
- `mifst`      : MIF-ST, structure-conditioned (needs AlphaFold PDB)
- `carp_640M`  : sequence-only (no structure)

Select via `MODEL_NAME` in the config cell.

## 1. Setup

In [ ]:
!nvidia-smi -L


In [ ]:
# sequence-models (microsoft/protein-sequence-models) ships the MIF, MIF-ST
# and CARP loaders. pandas + scipy are runtime requirements of sequence-models
# but not declared in its setup.py, so install them explicitly.
# biopython parses AlphaFold PDB files for MIF/MIF-ST.
#
# DO NOT pin numpy<2 here: Colab's preinstalled torch / scipy / pandas are
# built against numpy 2.x, and forcing a downgrade triggers
#   ValueError: numpy.dtype size changed, may indicate binary incompatibility
# at import time. (The numpy<2 pin is correct for the ESM2 cells because
# fair-esm 2.0.0 needs it — that's a separate runtime.)
!pip install -q sequence-models pandas scipy biopython requests tqdm

In [ ]:
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/LBDillon/decoding-design-bias.git'
REPO_DIR = '/content/decoding-design-bias'
DATASET  = '/content/main_plus_r2_r3_scored_filterC_v4.csv'  # edit if needed

DRIVE_OUT_DIR = '/content/drive/MyDrive/decoding-design-bias/outputs'

# Choose ONE model per runtime — switch via this constant.
MODEL_NAME = 'mif'  # 'mif' | 'mifst' | 'carp_640M'
OUTPUT     = f'{DRIVE_OUT_DIR}/{MODEL_NAME}_masked_scores.csv'
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

PDB_CACHE       = '/content/pdbs'
MASK_BATCH_SIZE = 8  # positions masked per forward pass; halved on OOM
os.makedirs(PDB_CACHE, exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=False)

assert os.path.exists(DATASET), DATASET
print('model    ->', MODEL_NAME)
print('output   ->', OUTPUT)


## 2. Load model

In [ ]:
import torch
import warnings; warnings.filterwarnings('ignore')

from sequence_models.pretrained import load_model_and_alphabet

assert torch.cuda.is_available(), 'Use a GPU runtime (Runtime -> Change runtime type -> GPU).'
device = torch.device('cuda')

model, collater = load_model_and_alphabet(MODEL_NAME)
model = model.to(device).eval()

STRUCTURE_CONDITIONED = {'mif', 'mifst'}
NEEDS_STRUCTURE = MODEL_NAME in STRUCTURE_CONDITIONED
print(f'{MODEL_NAME} loaded on {device}; needs structure: {NEEDS_STRUCTURE}')

# ----------------------------------------------------------------------
# Resolve the mask token id used by sequence-models for this model.
#
# Layouts confirmed locally on sequence-models 1.x (2026-05-26):
#   MIF / MIF-ST: collater = StructureCollater
#                   -> .sequence_collater = SimpleCollater
#                        -> .tokenizer = Tokenizer
#                             -> .mask_id (int), .alphabet (str with '#')
#   CARP:         collater = SimpleCollater (or wrapping equivalent)
#                   -> .tokenizer.mask_id / .alphabet
#
# This walks the collater 3 levels deep through any plausible wrapper
# attribute, prefers explicit `mask_id` integers over alphabet symbol
# lookups, and prints a diagnostic dump if discovery fails.
#
# If discovery still fails on a future sequence-models version, set
# MASK_ID_OVERRIDE below.
# ----------------------------------------------------------------------

MASK_ID_OVERRIDE = None  # set to an int if auto-discovery fails

MASK_SYMBOLS    = ('#', '<mask>', 'MASK', '?')
ATTR_CANDIDATES = ('mask_idx', 'mask_id', 'mask_token_id', 'mask_token', '_mask_idx')
WRAPPER_ATTRS   = ('tokenizer', 'tokeniser', 'sequence_collater', 'collater',
                   'alphabet', 'vocab', '_alphabet', '_vocab', '_tokenizer')


def _check(obj, label, found):
    if obj is None:
        return
    for attr in ATTR_CANDIDATES:
        if not hasattr(obj, attr):
            continue
        v = getattr(obj, attr)
        if isinstance(v, int):
            found.append((f'{label}.{attr}', v))
        elif isinstance(v, str):
            alpha = getattr(obj, 'alphabet', None)
            if alpha is not None:
                try:
                    found.append((f'{label}.{attr}_via_alphabet', alpha.index(v)))
                except (ValueError, AttributeError):
                    pass
    alpha = getattr(obj, 'alphabet', None)
    if alpha is None:
        return
    for sym in MASK_SYMBOLS:
        try:
            idx = alpha.index(sym)
            found.append((f"{label}.alphabet['{sym}']", idx))
            break
        except (ValueError, AttributeError):
            continue


def _resolve_mask_id():
    if MASK_ID_OVERRIDE is not None:
        return int(MASK_ID_OVERRIDE)

    found = []
    seen = set()
    queue = [(collater, 'collater', 0)]
    while queue:
        obj, label, depth = queue.pop(0)
        if id(obj) in seen or depth > 3:
            continue
        seen.add(id(obj))
        _check(obj, label, found)
        if depth < 3:
            for w in WRAPPER_ATTRS:
                child = getattr(obj, w, None)
                if child is not None and id(child) not in seen:
                    queue.append((child, f'{label}.{w}', depth + 1))

    if found:
        named = [t for t in found if '.alphabet[' not in t[0]]
        chosen = (named or found)[0]
        print(f'mask id discovery -> {chosen[0]} = {chosen[1]}')
        for src_label, idx in found[1:]:
            print(f'                also: {src_label} = {idx}')
        return int(chosen[1])

    print('mask id discovery FAILED. Diagnostic dump of the collater:')
    print(f'  collater type:   {type(collater).__name__}')
    print(f'  collater attrs:  '
          + str(sorted(a for a in dir(collater) if not a.startswith("__"))[:60]))
    tk = getattr(collater, 'tokenizer', None) or getattr(
        getattr(collater, 'sequence_collater', None), 'tokenizer', None
    )
    if tk is not None:
        print(f'  tokenizer type:  {type(tk).__name__}')
        print(f'  tokenizer attrs: '
              + str(sorted(a for a in dir(tk) if not a.startswith("__"))[:60]))
        if hasattr(tk, 'alphabet'):
            print(f'  tokenizer.alphabet: {tk.alphabet!r}')
    raise RuntimeError(
        'could not locate mask token id; set MASK_ID_OVERRIDE = <int> above '
        'using the diagnostic dump, then re-run this cell.'
    )


MASK_ID = _resolve_mask_id()
print(f'using mask token id = {MASK_ID}')

## 3. PDB fetcher (MIF / MIF-ST only) + sequence cleaning

In [ ]:
import requests, math
import pandas as pd
import numpy as np
import torch.nn.functional as F

VALID_STANDARD_AA = set('ACDEFGHIKLMNPQRSTVWY')

AF_URL_TEMPLATES = [
    'https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v6.pdb',
    'https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v4.pdb',
    'https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v3.pdb',
]

def fetch_pdb(entry, cache_dir=PDB_CACHE):
    for tmpl in AF_URL_TEMPLATES:
        url = tmpl.format(uid=entry)
        version = url.rsplit('model_', 1)[-1].replace('.pdb', '')
        local = os.path.join(cache_dir, f'AF-{entry}-F1-model_{version}.pdb')
        if os.path.exists(local):
            return local
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                with open(local, 'wb') as fh:
                    fh.write(r.content)
                return local
        except Exception:
            continue
    return None


def clean_sequence(seq):
    if seq is None or (isinstance(seq, float) and math.isnan(seq)):
        return '', 'missing'
    s = str(seq).strip().upper().replace(' ', '').replace('\n', '').replace('\r', '')
    if s.endswith('*'):
        s = s[:-1]
    bad = ''.join(sorted(set(s) - VALID_STANDARD_AA))
    return s, bad


def classify(seq_clean, bad):
    if not seq_clean:
        return 'empty_sequence'
    if bad:
        return 'nonstandard_amino_acid'
    return 'included'


In [ ]:
#@title PDB-MODE (R3.3) — score on experimental PDB chains instead of AlphaFold
#@markdown Toggle ON to score the experimental-structure subset. First upload
#@markdown **pdb_scoring_inputs.csv** and unzip **pdb_chain_structs.zip** to
#@markdown `/content/`. Run this cell AFTER the config/fetch_pdb cell and BEFORE
#@markdown the validation/scoring cells. Leave OFF to use AlphaFold (default).
PDB_MODE = False  #@param {type:"boolean"}
if PDB_MODE:
    import os, pandas as pd
    DATASET = "/content/pdb_scoring_inputs.csv"   # Entry, pdb_id, pdb_chain(=A), sequence(=chain), chain_pdb_path
    assert os.path.exists(DATASET), "Upload pdb_scoring_inputs.csv to /content/"
    _CHAINDIR = "/content/pdb_chain_structs"
    _pdb_df = pd.read_csv(DATASET)
    _pmap = {r.Entry: os.path.join(_CHAINDIR, os.path.basename(str(r.chain_pdb_path)))
             for r in _pdb_df.itertuples()}
    def fetch_pdb(entry, *args, **kwargs):   # override: local single-chain experimental PDB
        p = _pmap.get(entry)
        return p if (p and os.path.exists(p)) else None
    # write/resume from a SEPARATE file so the AlphaFold run's checkpoint isn't
    # reused (otherwise resume sees the AF rows as 'already scored' -> remaining 0)
    try:
        OUTPUT = os.path.splitext(OUTPUT)[0] + "_pdb.csv"
        print("OUTPUT ->", OUTPUT)
    except NameError:
        print("WARNING: OUTPUT not defined yet — run the config cell ABOVE this one first.")
    print(f"PDB-MODE ON — {len(_pmap)} experimental-structure inputs; DATASET -> {DATASET}")
    print("'sequence' is the resolved PDB chain; structures are single-chain (chain 'A').")
    print("Scores cover the resolved region; compare to AF2 scores per-residue (R3.3).")
else:
    print("PDB-MODE OFF — using AlphaFold structures (default).")


## 4. Masked-marginal scoring (the fix)

In [ ]:
# For MIF / MIF-ST we need structure features (process_coords output + backbone coords).
# Both parse_PDB and process_coords live in sequence_models.pdb_utils, NOT
# sequence_models.utils — that import path was wrong in the master notebook.
if NEEDS_STRUCTURE:
    from sequence_models.pdb_utils import parse_PDB, process_coords


def _prep_inputs_mif(seq, pdb_path, chain=None):
    """Return (src, nodes, edges, connections, edge_mask) all on device."""
    coords, _wt, _valid = parse_PDB(pdb_path, chain=chain)
    if len(coords) != len(seq):
        raise ValueError(f'sequence length {len(seq)} != structure length {len(coords)}')
    coords_dict = {'N': coords[:, 0], 'CA': coords[:, 1], 'C': coords[:, 2]}
    dist, omega, theta, phi = process_coords(coords_dict)
    batch = [[
        seq,
        torch.tensor(dist, dtype=torch.float),
        torch.tensor(omega, dtype=torch.float),
        torch.tensor(theta, dtype=torch.float),
        torch.tensor(phi, dtype=torch.float),
    ]]
    src, nodes, edges, connections, edge_mask = collater(batch)
    return (
        src.to(device), nodes.to(device), edges.to(device),
        connections.to(device), edge_mask.to(device),
    )


def _prep_inputs_carp(seq):
    src = collater([[seq]])[0].to(device)
    return src


def _forward_logits(src, *args):
    """Single forward pass; returns (L, vocab) log-softmax tensor on device."""
    with torch.no_grad():
        if NEEDS_STRUCTURE:
            nodes, edges, connections, edge_mask = args
            out = model(src, nodes, edges, connections, edge_mask, result='logits')[0]
        else:
            out = model(src, repr_layers=[], logits=True)['logits'][0]
    return F.log_softmax(out, dim=-1)


@torch.no_grad()
def masked_marginal_score(seq, pdb_path=None, mask_batch_size=MASK_BATCH_SIZE):
    """Compute proper masked-marginal PLL: for each position i, mask only that
    position, forward pass, read log p(WT_i | masked context).

    Batches `mask_batch_size` positions per forward pass by replicating the
    input tensor and masking position i of replica j. This is the same
    pattern as ESM2's score_esm2_sequence_batched (cell 23 of the master
    notebook).

    Verified locally 2026-05-26 against the single-position reference loop
    (diff = 0.00e+00 on a 42-residue test protein).
    """
    if NEEDS_STRUCTURE:
        src, nodes, edges, connections, edge_mask = _prep_inputs_mif(seq, pdb_path)
    else:
        src = _prep_inputs_carp(seq)
        nodes = edges = connections = edge_mask = None

    L = len(seq)
    targets = src[0, :L].clone()
    logp_chunks = []

    for start in range(0, L, mask_batch_size):
        end = min(start + mask_batch_size, L)
        B = end - start
        # Replicate the input B times, mask position (start + j) in replica j.
        batch_src = src.repeat(B, 1).clone()
        rows = torch.arange(B, device=device)
        cols = torch.arange(start, end, device=device)
        batch_src[rows, cols] = MASK_ID

        if NEEDS_STRUCTURE:
            batch_nodes       = nodes.repeat(B, *[1] * (nodes.dim() - 1))
            batch_edges       = edges.repeat(B, *[1] * (edges.dim() - 1))
            batch_connections = connections.repeat(B, *[1] * (connections.dim() - 1))
            batch_edge_mask   = edge_mask.repeat(B, *[1] * (edge_mask.dim() - 1))
            with torch.inference_mode():
                out = model(batch_src, batch_nodes, batch_edges,
                            batch_connections, batch_edge_mask, result='logits')
                log_probs = F.log_softmax(out, dim=-1)
        else:
            with torch.inference_mode():
                out = model(batch_src, repr_layers=[], logits=True)['logits']
                log_probs = F.log_softmax(out, dim=-1)

        for j in range(B):
            pos = start + j
            picked = log_probs[j, pos, targets[pos]]
            logp_chunks.append(picked.detach().cpu())
        del out, log_probs, batch_src
        torch.cuda.empty_cache()

    logp = torch.stack(logp_chunks)
    return {
        f'{MODEL_NAME}_score':     float(logp.mean().item()),
        f'{MODEL_NAME}_sum':       float(logp.sum().item()),
        f'{MODEL_NAME}_positions': int(logp.numel()),
        'scored_length':           int(logp.numel()),
    }


def _score_single_position_reference(seq, pdb_path=None):
    """Slow loop: mask one position at a time (no batching). Used to validate
    the batched implementation."""
    if NEEDS_STRUCTURE:
        src, nodes, edges, connections, edge_mask = _prep_inputs_mif(seq, pdb_path)
    else:
        src = _prep_inputs_carp(seq)
        nodes = edges = connections = edge_mask = None

    L = len(seq)
    targets = src[0, :L].clone()
    logp = []
    for i in range(L):
        one = src.clone()
        one[0, i] = MASK_ID
        with torch.inference_mode():
            if NEEDS_STRUCTURE:
                out = model(one, nodes, edges, connections, edge_mask, result='logits')[0]
            else:
                out = model(one, repr_layers=[], logits=True)['logits'][0]
            lp = F.log_softmax(out, dim=-1)
        logp.append(float(lp[i, targets[i]].item()))
    return {
        f'{MODEL_NAME}_score':     sum(logp) / len(logp) if logp else float('nan'),
        f'{MODEL_NAME}_sum':       sum(logp),
        f'{MODEL_NAME}_positions': len(logp),
        'scored_length':           len(logp),
    }


def score_with_oom_retry(seq, pdb_path=None, mask_batch_size=MASK_BATCH_SIZE):
    current = max(1, int(mask_batch_size))
    while True:
        try:
            return masked_marginal_score(seq, pdb_path, current), None
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if current == 1:
                return None, f'out_of_memory: L={len(seq)} mask_batch=1'
            current = max(1, current // 2)
            print(f'  OOM; retry with mask_batch_size={current}')
        except Exception as exc:
            return None, repr(exc)

## 5. Validation: batched score must equal single-position reference

In [ ]:
import csv, time

with open(DATASET) as fh:
    rows = list(csv.DictReader(fh))

# Pick three short test proteins (CARP doesn't need a PDB; MIF/MIF-ST do).
test_short = [r for r in rows if r.get('sequence') and 30 <= len(r['sequence']) <= 80][:3]
print(f'validation set: {len(test_short)} proteins')

TOLERANCE = 1e-4
passes = 0
for r in test_short:
    entry = r['Entry']
    seq, bad = clean_sequence(r.get('sequence', ''))
    if classify(seq, bad) != 'included':
        print(entry, 'skipped:', bad); continue
    pdb = fetch_pdb(entry) if NEEDS_STRUCTURE else None
    if NEEDS_STRUCTURE and pdb is None:
        print(entry, 'no PDB'); continue
    try:
        ref = _score_single_position_reference(seq, pdb)
        bat = masked_marginal_score(seq, pdb, mask_batch_size=MASK_BATCH_SIZE)
    except Exception as exc:
        print(entry, 'failed:', exc); continue
    diff = abs(ref[f'{MODEL_NAME}_score'] - bat[f'{MODEL_NAME}_score'])
    ok = diff < TOLERANCE
    passes += int(ok)
    print(f'{entry}  L={len(seq):3d}  ref={ref[f"{MODEL_NAME}_score"]:.6f}  '
          f'batched={bat[f"{MODEL_NAME}_score"]:.6f}  diff={diff:.2e}  {"OK" if ok else "MISMATCH"}')

assert passes == len(test_short), 'batched != reference; investigate before full run'
print(f'\nvalidation passed: batched implementation matches reference within {TOLERANCE}')


## 6. Full run with resume

In [ ]:
from tqdm.auto import tqdm

already = set()
if os.path.exists(OUTPUT):
    with open(OUTPUT) as fh:
        for row in csv.DictReader(fh):
            if row.get('Entry'):
                already.add(row['Entry'])
    print('resuming, already scored:', len(already))

todo = [r for r in rows if r['Entry'] not in already]
todo.sort(key=lambda r: len(r.get('sequence', '')))  # shortest first
print('remaining:', len(todo))

open_mode = 'a' if already else 'w'
score_col     = f'{MODEL_NAME}_score'
sum_col       = f'{MODEL_NAME}_sum'
positions_col = f'{MODEL_NAME}_positions'
FIELDS = [
    'Entry', 'species', 'domain',
    score_col, sum_col, positions_col,
    'scored_length', 'dataset_length',
    'sequence_filter_status', 'error',
]

with open(OUTPUT, open_mode, newline='') as out:
    w = csv.DictWriter(out, fieldnames=FIELDS)
    if open_mode == 'w':
        w.writeheader()

    counts = {'ok': 0, 'skipped_nonstd': 0, 'missing_pdb': 0, 'oom': 0, 'error': 0, 'empty': 0}
    t_start = time.time()
    for r in tqdm(todo, desc=MODEL_NAME):
        entry = r['Entry']
        rec = {
            'Entry': entry,
            'species': r.get('species', ''),
            'domain':  r.get('domain', ''),
            score_col: '', sum_col: '', positions_col: 0,
            'scored_length': 0,
            'dataset_length': len(r.get('sequence', '')),
            'sequence_filter_status': '',
            'error': '',
        }
        seq, bad = clean_sequence(r.get('sequence', ''))
        status = classify(seq, bad)
        rec['sequence_filter_status'] = status
        if status != 'included':
            if status == 'empty_sequence':
                counts['empty'] += 1
            else:
                counts['skipped_nonstd'] += 1
                rec['error'] = f'bad_chars={bad}'
            w.writerow(rec); out.flush(); continue

        pdb_path = None
        if NEEDS_STRUCTURE:
            pdb_path = fetch_pdb(entry)
            if pdb_path is None:
                rec['error'] = 'missing_pdb'
                counts['missing_pdb'] += 1
                w.writerow(rec); out.flush(); continue

        res, err = score_with_oom_retry(seq, pdb_path, mask_batch_size=MASK_BATCH_SIZE)
        if res is None:
            rec['error'] = err or 'unknown_error'
            if err and 'out_of_memory' in err:
                counts['oom'] += 1
            else:
                counts['error'] += 1
            w.writerow(rec); out.flush(); continue

        rec.update(res)
        w.writerow(rec); out.flush()
        counts['ok'] += 1
        if counts['ok'] % 50 == 0:
            torch.cuda.empty_cache()

print('done in', round(time.time() - t_start, 1), 's', counts)


## 7. Quick look

In [ ]:
import pandas as pd
df = pd.read_csv(OUTPUT)
print(df.shape)
print(f'non-null {score_col}:', df[score_col].notna().sum())
df[[score_col, sum_col, positions_col, 'scored_length']].describe()
